# PART 4: RAG, User Interface and Web Analytics

The web application is `web_app.py` (run it with `python web_app.py`). This notebook measures the parts of it that we discuss in the report: the search engine against the one of Part 3, the RAG pipeline against the baseline, and the analytics collected from simulated traffic.

In [1]:
import json, os, sys, time

import numpy as np
import pandas as pd

sys.path.append("../..")  # repository root

from myapp.search.load_corpus import load_corpus
from myapp.search.search_engine import SearchEngine, SearchFilters
from myapp.search import algorithms as part3
from myapp.search.evaluation import precision_at_k, avg_precision_at_k, ndcg_at_k
from myapp.generation.rag import BaselineRAG, ImprovedRAG

pd.options.display.float_format = "{:.3f}".format
pd.options.display.max_colwidth = 120

start = time.perf_counter()
corpus = load_corpus("../../data/fashion_products_dataset.json")
loaded = time.perf_counter()
engine = SearchEngine(corpus)
built = time.perf_counter()
print(f"Corpus: {len(corpus)} documents, loaded in {loaded - start:.1f} s")
print(f"Search engine index: {len(engine.index)} terms, built in {built - loaded:.1f} s")

Corpus: 28080 documents, loaded in 10.9 s
Search engine index: 5999 terms, built in 10.4 s


## 1. Search engine: Part 3 against Part 4

Part 3 ranks with our score on the terms processed in Part 1. Part 4 uses the same score, but joins spelling variants ("T-Shirt" / "tshirt") and does not index negative attributes ("Hooded: No"). We evaluate both with the relevance labels of Part 2.

In [2]:
with open("../part_1/processed_docs.json", encoding="utf-8") as f:
    docs = json.load(f)
doc_store = {d["pid"]: d for d in docs}
p3_index, p3_idf, p3_lengths, p3_avg = part3.create_bm25_index(docs)
p3_mean_rating = np.mean([d["average_rating"] for d in docs if d["average_rating"] is not None])

def part3_search(query):
    terms = part3.build_terms(query)
    pids = part3.conjunctive_match(terms, p3_index)
    bm25 = part3.score_bm25(terms, pids, p3_index, p3_idf, p3_lengths, p3_avg)
    return part3.rank(part3.score_custom(terms, bm25, doc_store, p3_mean_rating))

def part4_search(query):
    response = engine.search(query, page_size=len(corpus))
    return [(hit.document.pid, hit.score) for hit in response.hits]

labels = pd.read_csv("../part_2/test_labels.csv")
relevant = {q: set(g[g["labels"] == 1]["pid"]) for q, g in labels.groupby("query_id")}
TEST_QUERIES = {3: "women cotton kurta", 4: "men running shoes", 5: "polo neck tshirt men",
                6: "winter hooded jacket men", 7: "white sneakers women"}

rows = []
for q, text in TEST_QUERIES.items():
    for system, search in [("Part 3", part3_search), ("Part 4", part4_search)]:
        ranking = search(text)
        y_true = np.array([int(pid in relevant[q]) for pid, _ in ranking])
        y_score = np.array([s for _, s in ranking])
        rows.append({"query_id": q, "system": system, "retrieved": len(ranking),
                     "relevant retrieved": int(y_true.sum()), "recall": y_true.sum() / len(relevant[q]),
                     "precision (all)": y_true.mean() if len(y_true) else 0,
                     "P@10": precision_at_k(y_true, y_score, 10), "AP@10": avg_precision_at_k(y_true, y_score, 10),
                     "NDCG@10": ndcg_at_k(y_true, y_score, 10, np.ones(len(relevant[q])))})
comparison = pd.DataFrame(rows)
comparison

,query_id,system,retrieved,relevant retrieved,recall,precision (all),P@10,AP@10,NDCG@10
0,3,Part 3,575,322,1.000,0.560,1.000,1.000,1.000
1,3,Part 4,575,322,1.000,0.560,1.000,1.000,1.000
2,4,Part 3,76,23,1.000,0.303,1.000,1.000,1.000
3,4,Part 4,76,23,1.000,0.303,1.000,1.000,1.000
4,5,Part 3,203,186,0.140,0.916,1.000,1.000,1.000
5,5,Part 4,1410,1326,1.000,0.940,1.000,1.000,1.000
6,6,Part 3,292,49,1.000,0.168,0.700,1.000,0.801
7,6,Part 4,111,49,1.000,0.441,1.000,1.000,1.000
8,7,Part 3,76,3,1.000,0.039,0.300,0.867,0.947
9,7,Part 4,75,3,1.000,0.040,0.300,0.722,0.871


In [3]:
comparison.groupby("system")[["recall", "precision (all)", "P@10", "AP@10", "NDCG@10"]].mean()

,recall,precision (all),P@10,AP@10,NDCG@10
system,,,,,
Part 3,0.828,0.397,0.800,0.973,0.950
Part 4,1.000,0.457,0.860,0.944,0.974


In [4]:
# Why do the results of Query 6 change? Number of retrieved jackets by their "Hooded" attribute
def hooded_values(pids):
    return pd.Series([(corpus[p].product_details or {}).get("Hooded", "not given") for p in pids]).value_counts()

pd.DataFrame({"Part 3": hooded_values([p for p, _ in part3_search(TEST_QUERIES[6])]),
              "Part 4": hooded_values([p for p, _ in part4_search(TEST_QUERIES[6])])}).fillna(0).astype(int)

,Part 3,Part 4
No,181,0
Yes,68,68
not given,43,43


In [5]:
# Speed: time per query (ms), averaged over 20 runs of the five test queries
def timed(fn, query, runs=20):
    times = []
    for _ in range(runs):
        t = time.perf_counter(); fn(query); times.append((time.perf_counter() - t) * 1000)
    return np.mean(times)

engine._ranking.cache_clear()
speed = []
for q, text in TEST_QUERIES.items():
    t = time.perf_counter(); engine.search(text); cold = (time.perf_counter() - t) * 1000
    speed.append({"query": text, "Part 3 (ms)": timed(part3_search, text),
                  "Part 4, first time (ms)": cold, "Part 4, cached (ms)": timed(engine.search, text)})
speed = pd.DataFrame(speed)
speed.loc["mean"] = speed.mean(numeric_only=True)
speed

,query,Part 3 (ms),"Part 4, first time (ms)","Part 4, cached (ms)"
0,women cotton kurta,9.015,13.011,0.549
1,men running shoes,1.314,1.471,0.105
2,polo neck tshirt men,4.460,32.454,1.590
3,winter hooded jacket men,5.301,2.868,0.155
4,white sneakers women,1.827,1.858,0.176
mean,NaN,4.384,10.332,0.515


## 2. Query understanding

Budgets written in the query become a price filter, and when no product has all the terms the engine relaxes the query to all the terms but one.

In [6]:
examples = ["men running shoes under 1500", "kurta for women below rs 600", "polo neck t-shirt men",
            "polo neck tshirt men", "red leather wallet men", "iphone charger"]
rows = []
for text in examples:
    r = engine.search(text)
    rows.append({"query": text, "terms": " ".join(r.terms), "price limit": r.price_from_query, "results": r.total,
                 "relaxed": r.relaxed,
                 "first result": " ".join(r.hits[0].document.title.split()) if r.hits else "-"})
pd.DataFrame(rows)

,query,terms,price limit,results,relaxed,first result
0,men running shoes under 1500,men run shoe,1500.000,74,False,Running Shoes For Men (Multicolor)
1,kurta for women below rs 600,kurta women,600.000,237,False,Women Solid Cotton Blend Straight Kurta (Purple)
2,polo neck t-shirt men,polo neck tshirt men,NaN,1410,False,Striped Men Polo Neck Blue T-Shirt
3,polo neck tshirt men,polo neck tshirt men,NaN,1410,False,Striped Men Polo Neck Blue T-Shirt
4,red leather wallet men,red leather wallet men,NaN,14,True,"Sunshopping Y- Back Suspenders for Men, Boys (Red)"
5,iphone charger,iphon charger,NaN,0,True,-


## 3. RAG: baseline against the improved pipeline

### 3.1 What each pipeline sends to the LLM

In [7]:
baseline, improved = BaselineRAG(), ImprovedRAG()
RAG_QUERIES = ["women cotton kurta", "men running shoes under 1500", "white sneakers women",
               "red leather wallet men", "iphone charger"]

def baseline_context(response):
    return "\n".join(f"- PID: {h.document.pid}, Title: {h.document.title}" for h in response.hits[:20])

def improved_context(response):
    return "\n".join(improved._describe(d) for d in improved._select_context([h.document for h in response.hits]))

rows = []
for text in RAG_QUERIES:
    r = engine.search(text)
    top20 = [h.document for h in r.hits[:20]]
    rows.append({"query": text, "results": r.total,
                 "baseline products": len(top20),
                 "baseline distinct": len({(" ".join(d.title.split()), d.brand) for d in top20}),
                 "baseline chars": len(baseline_context(r)),
                 "improved products": len(improved._select_context([h.document for h in r.hits])),
                 "improved chars": len(improved_context(r))})
pd.DataFrame(rows)

,query,results,baseline products,baseline distinct,baseline chars,improved products,improved chars
0,women cotton kurta,575,20,19,1637,8,2456
1,men running shoes under 1500,74,20,18,1392,8,1547
2,white sneakers women,75,20,13,1611,8,2939
3,red leather wallet men,14,14,14,1051,8,2071
4,iphone charger,0,0,0,0,0,0


In [8]:
# Example of the context of each pipeline for the same query
r = engine.search("men running shoes under 1500")
print("BASELINE (first 5 of 20 lines)\n" + "\n".join(baseline_context(r).splitlines()[:5]))
print("\nIMPROVED (first 3 of", len(improved_context(r).splitlines()), "lines)\n" + "\n".join(improved_context(r).splitlines()[:3]))

BASELINE (first 5 of 20 lines)
- PID: SHOESS9ZEZ6TEQUJ, Title: Running Shoes For Men  (Multicolor)
- PID: SHOFN3EJVGC9VXYT, Title: RS-7200 Running Shoes For Men  (Navy)
- PID: SHOESS9ZGMH7CURM, Title: Running Shoes For Men  (Multicolor)
- PID: SHOFN3EJHXHGSGUC, Title: RS-7250 Running Shoes For Men  (Black)
- PID: SHOFN3EFNYGBKZT7, Title: RS-7050 Running Shoes For Men  (Blue, Navy)

IMPROVED (first 3 of 8 lines)
- [SHOESS9ZEZ6TEQUJ] Running Shoes For Men (Multicolor) | brand: Vector | Rs 1440 (9% off) | 4.3/5 | in stock | Ideal For: Men; Color: Multicolor; Occasion: Sports; Outer Material: Mesh
- [SHOFN3EJVGC9VXYT] RS-7200 Running Shoes For Men (Navy) | brand: Vector | Rs 990 (9% off) | 3.9/5 | in stock | Ideal For: Men; Color: Navy; Occasion: Sports; Outer Material: Fabric | sports shoes
- [SHOFN3EJHXHGSGUC] RS-7250 Running Shoes For Men (Black) | brand: Vector | Rs 990 (9% off) | 3.9/5 | in stock | Ideal For: Men; Color: Black; Occasion: Sports; Outer Material: Fabric | sports shoes


### 3.2 Answers

The LLM is called through the Groq API (`GROQ_API_KEY` in the `.env` file). Without a key, only the answers that the improved pipeline decides without the LLM are shown.

In [9]:
answers = []
for text in RAG_QUERIES:
    r = engine.search(text)
    limits = [p for p in (r.filters.max_price, r.price_from_query) if p is not None]
    b = baseline.generate_response(text, r.hits)
    i = improved.generate_response(text, r.hits, query_terms=r.terms, max_price=min(limits, default=None),
                                   relaxed=r.relaxed)
    answers.append({"query": text, "baseline": b, "improved": i})
    print("=" * 100)
    print(f"QUERY: {text}   ({r.total} results, relaxed={r.relaxed})")
    print(f"\n--- Baseline [{b.status}]\n{b.text.strip()}")
    print(f"\n--- Improved [{i.status}, LLM used: {i.used_llm}]\n{i.text.strip()}")

QUERY: women cotton kurta   (575 results, relaxed=False)

--- Baseline [unavailable]
RAG is not available. Check your credentials (.env file) or account limits.

--- Improved [unavailable, LLM used: False]
RAG is not available. Check your credentials (.env file) or account limits.
QUERY: men running shoes under 1500   (74 results, relaxed=False)

--- Baseline [unavailable]
RAG is not available. Check your credentials (.env file) or account limits.

--- Improved [unavailable, LLM used: False]
RAG is not available. Check your credentials (.env file) or account limits.
QUERY: white sneakers women   (75 results, relaxed=False)

--- Baseline [unavailable]
RAG is not available. Check your credentials (.env file) or account limits.

--- Improved [unavailable, LLM used: False]
RAG is not available. Check your credentials (.env file) or account limits.
QUERY: red leather wallet men   (14 results, relaxed=True)

--- Baseline [unavailable]
RAG is not available. Check your credentials (.env file) 

In [10]:
# Grounding check: are the products cited in each answer among the products retrieved?
rows = []
for a in answers:
    for name in ["baseline", "improved"]:
        ans = a[name]
        rows.append({"query": a["query"], "pipeline": name, "status": ans.status,
                     "cited pids": len(ans.cited_pids),
                     "cited and retrieved": sum(p in ans.context_pids for p in ans.cited_pids)})
pd.DataFrame(rows)

,query,pipeline,status,cited pids,cited and retrieved
0,women cotton kurta,baseline,unavailable,0,0
1,women cotton kurta,improved,unavailable,0,0
2,men running shoes under 1500,baseline,unavailable,0,0
3,men running shoes under 1500,improved,unavailable,0,0
4,white sneakers women,baseline,unavailable,0,0
5,white sneakers women,improved,unavailable,0,0
6,red leather wallet men,baseline,unavailable,0,0
7,red leather wallet men,improved,no_good_products,0,0
8,iphone charger,baseline,unavailable,0,0
9,iphone charger,improved,no_results,0,0


## 4. Web analytics on simulated traffic

`simulate_traffic.py` imitates 40 visitors during one week through the Flask application (see the file for the assumptions). The tables below are the data shown in the dashboard.

In [11]:
from simulate_traffic import SimClock, simulate
from datetime import datetime
from myapp.analytics.analytics_data import AnalyticsData
from web_app import create_app

clock = SimClock(datetime(2026, 11, 16))
analytics = AnalyticsData(clock=clock)
app = create_app(analytics=analytics, engine=engine, rag=ImprovedRAG())
simulate(app, clock, visitors=40, seed=7)
summary = analytics.summary(corpus)
pd.Series(summary["kpis"], name="value").to_frame()

,value
visitors,40.000
sessions,64.000
requests,1151.000
queries,163.000
unique_queries,30.000
clicks,355.000
external_clicks,45.000
ctr,0.890
zero_result_rate,0.037
relaxed_rate,0.049


In [12]:
tables = {
    "clicks by rank": pd.DataFrame(summary["clicks_by_rank"][:10], columns=["rank", "clicks"]).set_index("rank").T,
    "dwell time": pd.DataFrame(summary["dwell_bins"], columns=["dwell", "clicks"]).set_index("dwell").T,
    "query length": pd.DataFrame(summary["query_length"], columns=["terms", "queries"]).set_index("terms").T,
    "queries per mission": pd.DataFrame(summary["queries_per_mission"], columns=["queries", "missions"]).set_index("queries").T,
}
for name, table in tables.items():
    print(name); display(table)

clicks by rank


rank,1,2,3,4,5,6,7,8,9,10
clicks,89,44,27,18,26,19,21,17,7,11


dwell time


dwell,< 10 s,10-30 s,30-60 s,1-2 min,2-5 min,> 5 min
clicks,27,148,103,58,18,1


query length


terms,1,2,3,4,5,6+
queries,47,39,31,26,20,0


queries per mission


queries,1,2,3
missions,45,35,16


In [13]:
pd.concat({"browser": pd.DataFrame(summary["browsers"], columns=["value", "sessions"]),
           "os": pd.DataFrame(summary["os"], columns=["value", "sessions"]),
           "device": pd.DataFrame(summary["devices"], columns=["value", "sessions"])})

value  sessions
browser 0   Chrome        47
        1   Safari        11
        2  Firefox         4
        3     Edge         2
os      0  Windows        33
        1  Android        20
        2      iOS         8
        3    macOS         3
device  0  Desktop        36
        1   Mobile        28

In [14]:
hours = pd.Series(summary["queries_by_hour"], name="queries")
print("Busiest hours:", hours.sort_values(ascending=False).head(4).to_dict())
display(pd.DataFrame(summary["top_queries"], columns=["query", "times"]))
display(pd.DataFrame(summary["zero_queries"], columns=["query without results", "times"]))
pd.DataFrame(summary["top_docs"][:5])

Busiest hours: {22: 27, 15: 16, 13: 16, 19: 13}


,query,times
0,jeans,12
1,tshirt,10
2,men slim jeans blue,10
3,sneakers,9
4,track pants men,9
5,sweatshirt women,9
6,polo neck t-shirt men,8
7,kurta,7
8,women full sleeve sweatshirt cotton,7
9,men slim fit jeans black,7


,query without results,times
0,iphone charger,6


,pid,clicks,title
0,TKPFZAZ4QCAZFAGA,8,"Solid Men Black, Dark Blue Track Pants"
1,JEAFESND4QWQUBZD,7,Slim Men Blue Jeans
2,SWSFMTNHEM5SNHVZ,6,Full Sleeve Solid Women Sweatshirt
3,TSHFUGKTXHZTUGCH,6,Striped Men Polo Neck Blue T-Shirt
4,JEAFEH2STJYPUEYQ,6,Regular Men Blue Jeans


In [15]:
# Click-through rate and first-click rank per query type: do relaxed and refined queries work worse?
queries = pd.DataFrame([q.__dict__ for q in analytics.queries.values()])
clicks = pd.DataFrame([c.__dict__ for c in analytics.clicks if c.kind == "details"])
first_click = clicks.dropna(subset=["query_id"]).sort_values("timestamp").groupby("query_id")["rank"].first()
queries["clicked"] = queries["query_id"].isin(first_click.index)
queries["first click rank"] = queries["query_id"].map(first_click)
queries["type"] = np.select([queries["n_results"] == 0, queries["relaxed"], queries["order_in_session"] > 1],
                            ["no results", "relaxed", "later query of the session"], "first query of the session")
queries.groupby("type").agg(queries=("query_id", "count"), ctr=("clicked", "mean"),
                            mean_first_click_rank=("first click rank", "mean"), mean_terms=("n_terms", "mean"))

,queries,ctr,mean_first_click_rank,mean_terms
type,,,,
first query of the session,57,0.895,3.157,1.509
later query of the session,92,0.935,2.465,3.283
no results,6,0.000,NaN,2.000
relaxed,8,1.000,1.125,2.750
